# 🌐 Lab 8: 멀티클라우드 통합 게이트웨이 — 라우팅 테스트

## 목적
**하나의 OpenAI 호환 엔드포인트**(`/multicloud/chat/completions`)로 **Azure 외부의 프론티어 모델까지**
통합 호출합니다. 게이트웨이(Azure APIM)를 기본 플랫폼으로 두되, 컨셉의 핵심은 **Azure 밖의 3사
— OpenAI · Google(Gemini) · Anthropic — 을 하나의 계약으로 통합**하는 것입니다. 클라이언트는
**`model` 필드의 prefix**(`provider/model`)만 바꾸면 되고, 게이트웨이가 프로바이더별 백엔드·인증
(Named Value)으로 라우팅합니다.

| model 필드 | 라우팅 | 인증 |
|---|---|---|
| `gpt-4.1-nano` (prefix 없음) | Azure OpenAI (기본/네이티브) | Managed Identity |
| `openai/gpt-4o-mini` | OpenAI 직접 | `{{openai-api-key}}` |
| `anthropic/claude-...` | Anthropic (키 3개 풀) | `{{anthropic-api-key-1..3}}` |
| `gemini/gemini-2.5-flash` | Google Gemini | `{{gemini-api-key-1}}` |

## 이 노트북의 동작
1. **구성된 프로바이더 자동 감지** — Azure(항상), Gemini(Lab 5 Named Value 존재 시),
   OpenAI(`.env` 키 존재 시), Anthropic(`.env` 키 3개 존재 시). 미구성 프로바이더는 **안내 후 건너뜁니다**.
2. 통합 API + 라우팅 정책을 **실제 배포**하고, 구성된 프로바이더를 **라이브 호출**로 검증합니다.
3. 정리까지 자동.

> 💡 이 환경에서는 보통 **Azure + Gemini** 가 라이브로 검증되고, 나머지(OpenAI·Anthropic)는 키를 넣으면 그대로 켜집니다.
> 세 외부 프로바이더는 모두 **OpenAI 호환 엔드포인트**를 제공하므로 동일한 방식으로 통합됩니다.
> 관측(구독별·프로바이더별 토큰)은 [Lab 10 노트북](../lab10-governance-observability/test-governance-observability.ipynb) 참고.

In [1]:
# ─── 환경 설정 + 헬퍼 ───
import os, time, json, subprocess, tempfile
import requests
from dotenv import load_dotenv
load_dotenv("../../.env", override=True)

def az(args):
    r = subprocess.run(["az"] + args, capture_output=True, text=True)
    return r.stdout.strip(), r.stderr.strip(), r.returncode
def az_json(args):
    out, _, rc = az(args)
    try: return json.loads(out) if (rc == 0 and out) else None
    except json.JSONDecodeError: return None

SUBSCRIPTION_ID = os.getenv("AZURE_SUBSCRIPTION_ID") or (az_json(["account","show","--query","id","-o","json"]) or "")
RESOURCE_GROUP  = os.getenv("RESOURCE_GROUP","")
APIM_NAME       = os.getenv("APIM_NAME","")
if not (RESOURCE_GROUP and APIM_NAME):
    apims = az_json(["apim","list","--query","[].{name:name,rg:resourceGroup}","-o","json"]) or []
    if apims:
        APIM_NAME = APIM_NAME or apims[0]["name"]; RESOURCE_GROUP = RESOURCE_GROUP or apims[0]["rg"]
APIM_URL        = os.getenv("APIM_URL") or f"https://{APIM_NAME}.azure-api.net"
DEPLOYMENT_NAME = os.getenv("DEPLOYMENT_NAME","gpt-4.1-nano")
AV = "api-version=2024-06-01-preview"
assert SUBSCRIPTION_ID and APIM_NAME and RESOURCE_GROUP, "❌ az login / APIM 설정 확인 필요"
ARM_BASE = (f"https://management.azure.com/subscriptions/{SUBSCRIPTION_ID}"
            f"/resourceGroups/{RESOURCE_GROUP}/providers/Microsoft.ApiManagement/service/{APIM_NAME}")
UNIFIED_PATH = "multicloud"
UNIFIED_URL  = f"{APIM_URL}/{UNIFIED_PATH}/chat/completions"

def arm(method, path, body=None, query=None):
    url = f"{ARM_BASE}{path}?{AV}"; args = ["rest","--method",method,"--url",url]
    if query: args += ["--query",query,"-o","json"]
    tmp = None
    if body is not None:
        tmp = tempfile.NamedTemporaryFile("w",suffix=".json",delete=False); json.dump(body,tmp); tmp.close()
        args += ["--headers","Content-Type=application/json","--body",f"@{tmp.name}"]
    out, err, rc = az(args)
    if tmp: os.unlink(tmp.name)
    return out, err, rc
def arm_exists(path):
    _, _, rc = arm("GET", path, query="name"); return rc == 0

# 호출용 키: .env 의 APIM_SUBSCRIPTION_KEY 우선, 없으면 built-in all-access(master)
SUB_KEY = os.getenv("APIM_SUBSCRIPTION_KEY","").strip().strip('"')
if not SUB_KEY or SUB_KEY.startswith("<"):
    out,_,_ = arm("POST","/subscriptions/master/listSecrets", query="primaryKey"); SUB_KEY = (out or "").strip().strip('"')

print("✅ 설정 완료")
print(f"   APIM: {APIM_NAME}  통합 엔드포인트: {UNIFIED_URL}")
print(f"   호출 키: {'확보' if SUB_KEY else '❌ 없음'}")


✅ 설정 완료
   APIM: apim-ai-gw-aigateway-20260716  통합 엔드포인트: https://apim-ai-gw-aigateway-20260716.azure-api.net/multicloud/chat/completions
   호출 키: 확보


---
## 1단계: 구성된 프로바이더 감지

각 프로바이더는 **Named Value(secret)** 로 키를 보관하고 정책에서 `{{...}}` 로 참조합니다(제미나이와 동일 패턴).
- **Azure OpenAI** — 키 불필요(Managed Identity). 항상 사용 가능.
- **Gemini** — Lab 5에서 등록한 `gemini-api-key-1` Named Value 재사용.
- **OpenAI / Anthropic** — `.env` 에 키가 있으면 Named Value 로 등록하고 활성화, 없으면 건너뜀.


In [5]:
# ─── 프로바이더 구성 감지 ───
# provider -> {backend_id, backend_url, nv(참조할 named value), env(키 환경변수), model(테스트 모델), auth}
CATALOG = {
    "azure":     {"backend_id":"aoai-eastus", "nv":None, "env":None, "auth":"mi",
                  "model": DEPLOYMENT_NAME},
    "openai":    {"backend_id":"openai-direct-backend", "backend_url":"https://api.openai.com/v1",
                  "nv":"openai-api-key", "env":"OPENAI_API_KEY", "auth":"bearer",
                  "model":"openai/gpt-4o-mini"},
    "anthropic": {"backend_id":"anthropic-backend-pool", "backend_url":"https://api.anthropic.com/v1",
                  "nv":"anthropic-api-key", "env":"ANTHROPIC_API_KEY", "auth":"pool",
                  "model":"anthropic/claude-3-5-haiku-20241022"},
    "gemini":    {"backend_id":"gemini-openai-compat", "backend_url":"https://generativelanguage.googleapis.com/v1beta/openai",
                  "nv":"gemini-api-key-1", "env":None, "auth":"bearer",
                  "model":"gemini/gemini-2.5-flash"},
}
def _clean(v): return v.strip().strip('"') if v else ""
def _valid(v): return bool(v) and not v.startswith("<")

configured, skipped = {}, {}
for p, c in CATALOG.items():
    if p == "azure":
        configured[p] = c; continue
    if c["auth"] == "pool":
        # 키 3개 수집: ANTHROPIC_API_KEY_1..3, 없으면 단일 ANTHROPIC_API_KEY 로 폴백
        # ⚠️ APIM 은 백엔드 credential 의 {{named-value}} 를 해석하지 않으므로 키를 raw 로 백엔드에 주입한다.
        keys = [k for k in (_clean(os.getenv(f'{c["env"]}_{i}')) for i in (1, 2, 3)) if _valid(k)]
        if not keys:
            single = _clean(os.getenv(c["env"]))
            if _valid(single): keys = [single]
        if keys:
            configured[p] = dict(c, _keys=keys)
        else:
            skipped[p] = f'{c["env"]}_1'
        continue
    env_key = _clean(os.getenv(c["env"])) if c["env"] else ""
    has_env = _valid(env_key)
    has_nv  = arm_exists(f"/namedValues/{c['nv']}") if c["nv"] else False
    if has_env or has_nv:
        configured[p] = dict(c, _env_val=env_key if has_env else None)
    else:
        skipped[p] = c["env"]

print("구성된 프로바이더:")
for p in configured:
    extra = f' (키 {len(configured[p]["_keys"])}개 풀)' if configured[p].get("_keys") else ""
    print(f"  ✅ {p:10s} → {configured[p]['model']}{extra}")
print("\n건너뛴 프로바이더(키 없음):")
for p, env in skipped.items(): print(f"  ⏭️  {p:10s} → .env 에 {env} 를 설정하면 활성화")

구성된 프로바이더:
  ✅ azure      → gpt-4.1-nano
  ✅ anthropic  → anthropic/claude-3-5-haiku-20241022 (키 3개 풀)
  ✅ gemini     → gemini/gemini-2.5-flash

건너뛴 프로바이더(키 없음):
  ⏭️  openai     → .env 에 OPENAI_API_KEY 를 설정하면 활성화


---
## 2단계: 백엔드·Named Value·통합 API 배포

구성된 프로바이더에 대해서만 백엔드(+필요 시 Named Value)를 만들고, `model` prefix 로 분기하는
라우팅 정책을 통합 API 에 적용합니다. Azure 는 요청을 배포 경로(`/deployments/{model}/...`)로,
그 외 프로바이더는 OpenAI 호환 경로(`/chat/completions`)로 재작성합니다.


In [6]:
# ─── 백엔드/NV 등록 + 통합 API & 라우팅 정책 배포 ───
# 1) 백엔드 + Named Value (구성된 외부 프로바이더)
CB_RULES = {"rules":[{
    "failureCondition":{"count":3,"errorReasons":["Server errors"],"interval":"PT10S",
        "statusCodeRanges":[{"min":429,"max":429},{"min":500,"max":503}]},
    "name":"providerCircuitBreaker","tripDuration":"PT30S","acceptRetryAfter":False}]}
for p, c in configured.items():
    if p == "azure": continue
    if c["auth"] == "pool":
        keys = c.get("_keys") or []
        # 키별 백엔드: 각각 다른 키를 credential 에 직접(raw) 주입 + Circuit Breaker
        # ⚠️ APIM 은 백엔드 credential 의 {{named-value}} 를 해석하지 않으므로 raw 키를 넣는다.
        for i, key in enumerate(keys, 1):
            arm("PUT", f'/backends/anthropic-backend-{i}', body={"properties":{
                "protocol":"http", "url":c["backend_url"],
                "credentials":{"header":{"Authorization":[f"Bearer {key}"]}},
                "circuitBreaker":CB_RULES}})
        # 여러 백엔드를 하나의 풀로 묶어 Round Robin 로드밸런싱
        if keys:
            services = [{"id":f"/backends/anthropic-backend-{i}","priority":1,"weight":1}
                        for i in range(1, len(keys)+1)]
            arm("PUT", "/backends/anthropic-backend-pool",
                body={"properties":{"type":"Pool","pool":{"services":services}}})
        continue
    arm("PUT", f"/backends/{c['backend_id']}", body={"properties":{"protocol":"http","url":c["backend_url"]}})
    if c.get("_env_val"):  # .env 키가 있으면 secret Named Value 로 등록
        arm("PUT", f"/namedValues/{c['nv']}", body={"properties":{
            "displayName":c["nv"], "secret":True, "value":c["_env_val"]}})
print("✅ 백엔드/Named Value 준비:", ", ".join(p for p in configured if p!="azure") or "(외부 없음)")

# 2) 라우팅 정책 (구성된 프로바이더 브랜치만 포함)
PARSE = ('<set-variable name="rawModel" value="@{var b=context.Request.Body?.As&lt;Newtonsoft.Json.Linq.JObject&gt;'
         '(preserveContent:true);return b!=null&amp;&amp;b[&quot;model&quot;]!=null?b[&quot;model&quot;].ToString():&quot;&quot;;}" />'
         '<set-variable name="provider" value="@{var m=(string)context.Variables[&quot;rawModel&quot;];'
         'return m.Contains(&quot;/&quot;)?m.Split(\'/\')[0]:&quot;azure&quot;;}" />'
         '<set-variable name="modelName" value="@{var m=(string)context.Variables[&quot;rawModel&quot;];'
         'return m.Contains(&quot;/&quot;)?m.Substring(m.IndexOf(\'/\')+1):m;}" />'
         '<set-body>@{var b=context.Request.Body.As&lt;Newtonsoft.Json.Linq.JObject&gt;(preserveContent:true);'
         'b[&quot;model&quot;]=(string)context.Variables[&quot;modelName&quot;];return b.ToString();}</set-body>')
def branch(p, c):
    cond = f'@((string)context.Variables[&quot;provider&quot;]==&quot;{p}&quot;)'
    if c["auth"] == "mi":
        inner = ('<set-backend-service backend-id="aoai-eastus" />'
                 '<authentication-managed-identity resource="https://cognitiveservices.azure.com" />'
                 '<rewrite-uri template="@(&quot;/deployments/&quot;+(string)context.Variables[&quot;modelName&quot;]+'
                 '&quot;/chat/completions?api-version=2025-04-01-preview&quot;)" />')
    elif c["auth"] == "pool":
        # 풀 백엔드가 라운드로빈으로 선택 → 각 백엔드 credential(raw 키)이 인증 주입 (정책에서 인증 헤더 불필요)
        inner = (f'<set-backend-service backend-id="{c["backend_id"]}" />'
                 f'<rewrite-uri template="/chat/completions" />')
    else:
        inner = (f'<set-backend-service backend-id="{c["backend_id"]}" />'
                 f'<set-header name="Authorization" exists-action="override"><value>@(&quot;Bearer &quot;+&quot;{{{{{c["nv"]}}}}}&quot;)</value></set-header>'
                 f'<rewrite-uri template="/chat/completions" />')
    return f'<when condition="{cond}">{inner}</when>'

branches = "".join(branch(p, c) for p, c in configured.items())
otherwise = ('<otherwise><return-response><set-status code="400" reason="Provider not configured" />'
             '<set-header name="Content-Type" exists-action="override"><value>application/json</value></set-header>'
             '<set-body>@(&quot;{\\&quot;error\\&quot;:\\&quot;provider &quot;+(string)context.Variables[&quot;provider&quot;]+&quot; not configured in this environment\\&quot;}&quot;)</set-body>'
             '</return-response></otherwise>')
POLICY = (f'<policies><inbound><base />{PARSE}<choose>{branches}{otherwise}</choose>'
          '</inbound><backend><base /></backend><outbound><base /></outbound><on-error><base /></on-error></policies>')

# 3) 통합 API + operation + 정책
arm("PUT", f"/apis/multicloud-openai", body={"properties":{
    "displayName":"Multicloud OpenAI", "path":UNIFIED_PATH, "protocols":["https"],
    "subscriptionRequired":True, "serviceUrl":"https://api.openai.com/v1"}})
arm("PUT", "/apis/multicloud-openai/operations/chat", body={"properties":{
    "displayName":"chat", "method":"POST", "urlTemplate":"/chat/completions", "templateParameters":[]}})
_, err, rc = arm("PUT", "/apis/multicloud-openai/policies/policy",
                 body={"properties":{"format":"rawxml","value":POLICY}})
print("✅ 통합 API 배포" if rc == 0 else f"❌ 정책 오류: {err[:300]}")
print("⏳ 게이트웨이 전파 40초..."); time.sleep(40)

✅ 백엔드/Named Value 준비: anthropic, gemini
✅ 통합 API 배포
⏳ 게이트웨이 전파 40초...


---
## 3단계: 라우팅 테스트 (동일 요청, `model` 만 변경)

**같은 OpenAI 포맷 요청**에서 `model` 필드만 바꿔 각 클라우드로 라우팅되는지 확인합니다.
구성된 프로바이더는 **200 + 응답**, 미구성 프로바이더는 게이트웨이가 **400 안내**를 반환합니다.

```python
# 클라이언트는 표준 OpenAI SDK 그대로 — base_url 만 통합 엔드포인트로
from openai import OpenAI
client = OpenAI(base_url="{APIM}/multicloud", api_key="<APIM_SUBSCRIPTION_KEY>")
client.chat.completions.create(model="gemini/gemini-2.5-flash", messages=[...])
```


In [ ]:
# ─── 라우팅 테스트 ───
class _Err:  # 예외 시 requests.Response 대용 (일시적 네트워크 오류 표시)
    def __init__(self, msg): self.text = msg
    def json(self): return {}

def call(model, attempts=3):
    last = None
    for a in range(attempts):
        try:
            r = requests.post(UNIFIED_URL,
                headers={"Content-Type":"application/json","Ocp-Apim-Subscription-Key":SUB_KEY},
                json={"model":model,"messages":[{"role":"user","content":"Reply with a short greeting."}],
                      "max_tokens":20}, timeout=60)
            return r.status_code, r
        except requests.exceptions.RequestException as e:
            last = e; time.sleep(2 * (a + 1))  # 일시적 SSL/연결 오류 → 백오프 후 재시도
    return 0, _Err(f"connection error after {attempts} tries: {str(last)[:100]}")

print("═"*72); print(" 구성된 프로바이더 (200 기대)"); print("═"*72)
results = {}
for p, c in configured.items():
    code_, r = call(c["model"]); results[p] = code_
    if code_ == 200:
        try:
            j = r.json()
            msg = ((j.get("choices") or [{}])[0].get("message", {}).get("content") or "").strip().replace("\n", " ")[:50]
            tok = j.get("usage", {}).get("total_tokens", "?")
        except ValueError:
            msg, tok = r.text[:50], "?"
        print(f"  ✅ {p:10s} {c['model']:52s} → 200 | tokens={tok} | \"{msg}\"")
    else:
        print(f"  ⚠️ {p:10s} {c['model']:52s} → {code_} | {r.text[:120]}")

if skipped:
    print("\n" + "═"*72); print(" 미구성 프로바이더 (400 안내 기대)"); print("═"*72)
    demo = {"openai":"openai/gpt-4o","anthropic":"anthropic/claude-3-5-haiku-20241022"}
    for p in skipped:
        code_, r = call(demo.get(p, f"{p}/x"))
        print(f"  🚫 {p:10s} → {code_} | {r.text[:90]}")

print("\n관찰 포인트")
print("  • 동일한 요청에서 model prefix 만으로 서로 다른 클라우드로 라우팅됨")
print("  • Azure 는 배포경로로, 그 외는 OpenAI 호환 경로로 게이트웨이가 재작성")
print("  • 미구성 프로바이더는 게이트웨이가 명확한 400 으로 차단 (백엔드 도달 전)")

════════════════════════════════════════════════════════════════════════
 구성된 프로바이더 (200 기대)
════════════════════════════════════════════════════════════════════════


SSLError: HTTPSConnectionPool(host='apim-ai-gw-aigateway-20260716.azure-api.net', port=443): Max retries exceeded with url: /multicloud/chat/completions (Caused by SSLError(SSLEOFError(8, '[SSL: UNEXPECTED_EOF_WHILE_READING] EOF occurred in violation of protocol (_ssl.c:1081)')))

---
## 4단계: 관측 연계

이 통합 API에 `llm-emit-token-metric`(Provider 차원)을 추가하면, **어떤 클라우드로 얼마나** 나갔는지
`customMetrics` 에서 프로바이더별로 분해됩니다. 구독별 격리·쿼터는 Lab 9, 프로바이더×구독 관측 대시보드는
Lab 10에서 다룹니다.

- 👉 구독별 거버넌스 실습: [Lab 9 노트북](../lab09-products-portal/test-products-portal.ipynb)
- 👉 프로바이더×구독 관측(App Insights KQL): [Lab 10 노트북](../lab10-governance-observability/test-governance-observability.ipynb)


---
## 정리
테스트로 만든 통합 API와 백엔드를 삭제합니다. (Lab 5 의 `gemini-api-key-*` Named Value 는 보존)


In [ ]:
# ─── 정리 ───
arm("DELETE", "/apis/multicloud-openai")
print("  🧹 통합 API 삭제")
# 노트북이 만든 백엔드 삭제 (기존 aoai-*/gemini-backend-* 는 건드리지 않음)
for p, c in configured.items():
    if p == "azure": continue
    if c["auth"] == "pool":
        arm("DELETE", "/backends/anthropic-backend-pool"); print("  🧹 백엔드 삭제: anthropic-backend-pool")
        for i in range(1, len(c.get("_keys") or [])+1):
            arm("DELETE", f"/backends/anthropic-backend-{i}"); print(f"  🧹 백엔드 삭제: anthropic-backend-{i}")
        continue
    if c["backend_id"] in ("gemini-openai-compat","openai-direct-backend"):
        arm("DELETE", f"/backends/{c['backend_id']}"); print(f"  🧹 백엔드 삭제: {c['backend_id']}")
# .env 로 새로 만든 외부 Named Value 삭제 (Lab 5 gemini-api-key-* 는 유지, anthropic 풀은 NV 미사용)
for p, c in configured.items():
    if c["auth"] == "pool": continue
    if c.get("_env_val") and c["nv"] in ("openai-api-key",):
        arm("DELETE", f"/namedValues/{c['nv']}"); print(f"  🧹 Named Value 삭제: {c['nv']}")
print("✅ 정리 완료 (gemini-api-key-* 는 보존)")

  🧹 통합 API 삭제


  🧹 백엔드 삭제: gemini-openai-compat
✅ 정리 완료 (gemini-api-key-* 는 보존)


---
## ✅ 요약

| 확인 | 내용 |
|---|---|
| **통합 엔드포인트** | 하나의 OpenAI 호환 `/multicloud/chat/completions` 로 멀티클라우드 호출 |
| **모델 prefix 라우팅** | `provider/model` 로 백엔드·인증을 게이트웨이가 선택 |
| **Named Value 인증** | 프로바이더 키를 secret Named Value(`{{...}}`)로 주입 (제미나이와 동일) |
| **Azure = Managed Identity** | 키 없이 MI 로 인증, 배포경로로 재작성 |
| **미구성 게이팅** | 키 없는 프로바이더는 게이트웨이가 400 으로 안전 차단 |

→ 다음: [Lab 9: 구독별 거버넌스 & Developer Portal](../lab09-products-portal/README.md) ·
[Lab 10: 관측 캡스톤](../lab10-governance-observability/README.md)
